# SwiftDub — Colab GPU Demo

Sets up SwiftDub on Colab GPU, downloads models, runs Wav2Lip / MuseTalk / LatentSync, and hosts the FastAPI service.

**Runtime:** GPU (T4+ recommended; LatentSync 1.5 needs ~8GB VRAM)

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Clone or upload SwiftDub
import os
if not os.path.exists('SwiftDub'):
    !git clone https://github.com/YOUR_USERNAME/SwiftDub.git
%cd SwiftDub
else:
    %cd SwiftDub

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg git
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q -r requirements.txt
!pip install -q pyngrok

In [ ]:
# Download sample data + models (start with wav2lip; add others as needed)
!python scripts/download_sample_data.py
!python scripts/download_models.py --model wav2lip
# Uncomment to benchmark all three (large download):
# !python scripts/download_models.py --model all --latentsync-version 1.5

In [ ]:
# Run pipeline demo on public LatentSync sample assets
!python scripts/run_pipeline_demo.py --model wav2lip
# Benchmark all installed models:
# !python scripts/run_pipeline_demo.py --all-models

In [ ]:
# Host FastAPI demo via ngrok (set NGROK_AUTHTOKEN in Colab secrets)
import threading
from pyngrok import ngrok
import uvicorn

try:
    from google.colab import userdata
    ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
except Exception:
    pass  # free tunnel still works with session limits

public_url = ngrok.connect(8000)
print('SwiftDub API:', public_url, '/docs')

def run_server():
    uvicorn.run('src.main:app', host='0.0.0.0', port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print('Server running. Test GET /health and POST /predict from the public URL.')